In [1]:
import os
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import logging
import sys
from datetime import datetime
import numpy as np
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

In [3]:

# 读取 Excel 文件中的两个表格
file_path = "./error_sn/error_train1.csv"
ok_data=pd.read_csv(file_path)
#sheet1 = pd.read_excel(file_path, sheet_name='DOA')
#sheet2 = pd.read_excel(file_path, sheet_name='RMA')

# 创建新的数据框并重命名列
new_df1 = ok_data[['sn', 'return_date']].copy()
new_df1.columns = ['sn', 'createtime']
# 拼接三个数据框
#combined_df = pd.concat([new_df1, new_df2], ignore_index=True)

# 去重
unique_df = new_df1.drop_duplicates()

# 输出结果

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("inverter_data_fetch.log"),
        logging.StreamHandler(sys.stdout)
    ]
)

# Database connection information
DATABASE_URL = (
    "postgresql+psycopg2://qj4807BZhPRqYOR6:8Vt0RAoceGkx8qFlajIU7czwajECYb"
    "@holo-cn-1fe0d3158320-cn-cz-thgndsj-d01-vpc-st.hologres.asoops.trinasolar.com:80/ts_bd_rt"
)

# 构造包含 vpv 和 ipv 的列名
vpv_columns = [f'vpv{i}' for i in range(1, 31)]  # vpv1 到 vpv30
ipv_columns = [f'ipv{i}' for i in range(1, 31)]  # ipv1 到 ipv30

# 将其他需要的列添加到列表中
COLUMNS_TO_SELECT = [
    'temperature', 'vac1', 'vac2', 'vac3', 
    'iac1', 'iac2', 'iac3', 'fac1', 'fac2','fac3', 'pac', 
    'eday', 'etotal', 'ttotal', 'outputpowerratio', 
    'activepower', 'reactivepower', 'inspectingpower', 
    'powerfactor', 'createtime', 'sn', 'switchstatus','errorcode'
] + vpv_columns + ipv_columns

# Columns to select
#COLUMNS_TO_SELECT = [
#    'sn', 'temperature', 'vpv1', 'vpv2', 'ipv1', 'ipv2',
#    'vac1', 'vac2', 'vac3', 'iac1', 'iac2', 'fac3',
#    'pac', 'eday', 'etotal', 'ttotal', 'switchstatus',
#    'createtime', 'outputpowerratio', 'activepower',
#    'reactivepower', 'inspectingpower', 'powerfactor'
#]
COLUMNS_STR = ', '.join(COLUMNS_TO_SELECT)

# Create database engine
def create_db_engine():
    try:
        engine = create_engine(
            DATABASE_URL,
            pool_size=50,         # Maximum number of connections in the pool
            max_overflow=5,       # Maximum overflow connections
            pool_timeout=30,      # Timeout for getting a connection from the pool
            pool_recycle=1800,    # Recycle connections after 1800 seconds
            pool_pre_ping=True    # Check connections' liveness
        )
        logging.info("Successfully created the database engine.")
        return engine
    except SQLAlchemyError as e:
        logging.error(f"Failed to create the database engine: {e}")
        sys.exit(1)

# Initialize the engine
engine = create_db_engine()

# Function to get unique SNs
def select_unique_sns(before_september_2024):
    try:
        sn_list = before_september_2024['sn'].unique().tolist()
        logging.info(f"Retrieved {len(sn_list)} unique SNs.")
        return sn_list
    except Exception as e:
        logging.error(f"Error retrieving unique SNs: {e}")
        return []

# Function to fetch data for a single SN
def fetch_data_for_sn(sn, createtime, batch_size=6000):
    total_rows = []
    try:
        # Convert createtime to Python datetime if it's a numpy.datetime64 or pandas.Timestam

        print("crate",createtime)
        createtime = pd.to_datetime(createtime, format='%Y%m%d')
        logging.info(f"Fetching data for SN: {sn} with createtime: {createtime}")

        sql_query = f"""
            WITH LatestTime AS (
                SELECT sn, MAX(createtime::timestamp) AS latest_time
                FROM ts_bd_ods.ods_power_inverter_data_di
                WHERE sn = :sn
                GROUP BY sn
            ),
            FilteredData AS (
                SELECT d.*
                FROM ts_bd_ods.ods_power_inverter_data_di AS d
                JOIN LatestTime AS l ON d.sn = l.sn
                WHERE d.createtime::timestamp 
                BETWEEN (CAST(:createtime AS timestamp) - INTERVAL '60 days') 
                AND (CAST(:createtime AS timestamp))
            )
            SELECT {COLUMNS_STR}
            FROM FilteredData
            ORDER BY createtime DESC;
        """

        with engine.connect() as connection:
            result = connection.execution_options(stream_results=True).execute(
                text(sql_query),
                {"sn": sn, "createtime": createtime}
            )

            # Initialize tqdm progress bar
            with tqdm(total=batch_size, desc=f"Fetching SN {sn} data", leave=False) as pbar:
                while True:
                    rows = result.fetchmany(batch_size)
                    if not rows:
                        break
                    total_rows.extend(rows)
                    pbar.update(len(rows))

        logging.info(f"Fetched {len(total_rows)} rows for SN: {sn}")

    except SQLAlchemyError as e:
        logging.error(f"Error fetching data for SN {sn}: {e}")
    except Exception as e:
        logging.error(f"Unexpected error for SN {sn}: {e}")

    return total_rows

# Main function to orchestrate data fetching
def main(unique_df):
    import time
    start_time = time.time()


        # Load data with createtime parsed as datetime
    #before_september_2024 = pd.read_csv("before_september_2024.csv", parse_dates=['createtime'])
    unique_df = unique_df.drop_duplicates(subset=['sn'])
    print(unique_df)


    sn_list = select_unique_sns(unique_df)

    all_data = []

    # Use ThreadPoolExecutor for concurrent data fetching
    with ThreadPoolExecutor(max_workers=10) as executor:
        # Submit all tasks
        future_to_sn = {
            executor.submit(
                fetch_data_for_sn, 
                sn, 
                unique_df.loc[unique_df['sn'] == sn, 'createtime'].values[0]
            ): sn
            for sn in sn_list
        }

        # Process completed tasks as they finish
        for future in as_completed(future_to_sn):
            sn = future_to_sn[future]
            try:
                data = future.result()
                all_data.extend(data)
            except Exception as e:
                logging.error(f"Error processing SN {sn}: {e}")

    # Convert all data to a DataFrame
    if all_data:
        df = pd.DataFrame(all_data, columns=COLUMNS_TO_SELECT)
        # Example: Save to CSV
        df.to_parquet("./ok_12sn.parquet", index=False)
        logging.info(f"Successfully fetched and saved {len(df)} rows of data.")
    else:
        logging.warning("No data was fetched.")

    end_time = time.time()
    logging.info(f"Data fetching completed in {end_time - start_time:.2f} seconds.")

if __name__ == "__main__":
    main(unique_df)


2025-01-21 10:24:08,792 [INFO] Successfully created the database engine.
                     sn  createtime
0  TP40KBT000B230331239    20250120
1  TP33KBT000B230411073    20250120
2025-01-21 10:24:08,805 [INFO] Retrieved 2 unique SNs.
crate 20250120
2025-01-21 10:24:08,822 [INFO] Fetching data for SN: TP40KBT000B230331239 with createtime: 2025-01-20 00:00:00
crate 20250120
2025-01-21 10:24:08,838 [INFO] Fetching data for SN: TP33KBT000B230411073 with createtime: 2025-01-20 00:00:00


Fetching SN TP33KBT000B230411073 data: 7110it [00:01, 4411.09it/s]                          

2025-01-21 10:24:36,482 [INFO] Fetched 7095 rows for SN: TP40KBT000B230331239


2025-01-21 10:24:36,510 [INFO] Fetched 7110 rows for SN: TP33KBT000B230411073


2025-01-21 10:24:36,756 [INFO] Successfully fetched and saved 14205 rows of data.
2025-01-21 10:24:36,758 [INFO] Data fetching completed in 27.96 seconds.
